# Regex for NLP: Practice Lab
**52 graded tasks** covering every part of the tutorial (Parts 1-11).

**How it works**
1. Run the *Setup + Grader* cell once.
2. For each task: read the **Rule**, write your answer where it says `# <- your regex` / `# TODO`, run the cell.
3. The cell prints ✅ PASS or ❌ FAIL and, for **every test case**, an `EXPECTED:` value (always 🟩 green) next to your `ACTUAL:` result (🟩 green if it matches, 🟥 red if it doesn't).
4. Run `scoreboard()` at the end (last cell) to see your progress.

Tips: patterns are graded as **raw strings**; unless the task says otherwise, matching uses `fullmatch` (whole string) or `finditer` (all matches). Part 4.3 and 6.1 need the optional module: `pip install regex`.

In [59]:
# ============ GRADER (run this cell first; no need to read it) ============
import re, sys, subprocess, html, unicodedata, time
try:
    import regex            # optional third-party module (Part 6 bonus)
except ImportError:
    regex = None

RESULTS = {}

# ---- Output style: EXPECTED is always green. ACTUAL is green if it matches EXPECTED, red if it doesn't. ----
_GREEN, _RED = "#1faa4b", "#e5383b"
_ANSI = {_GREEN: "\033[1;32m", _RED: "\033[1;31m"}

def _in_notebook():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        return ip is not None and getattr(ip, "kernel", None) is not None
    except Exception:
        return False

_NA = object()          # "nothing to compare yet" (empty / invalid pattern)

def _mm(b):             # bool -> text for match / no-match checks
    return "MATCH" if b else "NO MATCH"

def _try(fn):
    try: return fn(), None
    except Exception as ex: return _NA, f"{type(ex).__name__}: {ex}"

def _chk(label, exp, got, err=None, fmt=repr, compact=None):
    ok = err is None and got is not _NA and got == exp
    e_s = fmt(exp)
    if err:            a_s = "⚠ " + err
    elif got is _NA:   a_s = "— (nothing to run yet)"
    else:              a_s = fmt(got)
    if compact is None:
        compact = len(label) + len(e_s) + len(a_s) <= 60
    return dict(label=label, exp=e_s, act=a_s, ok=ok, compact=compact)

def _rep(tid, checks, error=None):
    ok = error is None and all(c["ok"] for c in checks)
    RESULTS[tid] = ok
    head = ("✅ PASS" if ok else "❌ FAIL") + f"   Task {tid}"
    if _in_notebook():
        from IPython.display import display, HTML
        esc = html.escape
        span = lambda t, col: f'<span style="color:{col};font-weight:bold">{esc(t)}</span>'
        L = [f'<b style="color:{_GREEN if ok else _RED}">{esc(head)}</b>']
        if error: L.append("   " + span("⚠ " + error, _RED))
        for c in checks:
            col = _GREEN if c["ok"] else _RED
            mark = span("✔" if c["ok"] else "✘", col)
            if c["compact"]:
                L.append(f'   {mark} {esc(c["label"])}   <b>EXPECTED:</b> {span(c["exp"], _GREEN)}   <b>ACTUAL:</b> {span(c["act"], col)}')
            else:
                L.append(f'   {mark} {esc(c["label"])}')
                L.append(f'        <b>EXPECTED:</b> {span(c["exp"], _GREEN)}')
                L.append(f'        <b>ACTUAL:</b>   {span(c["act"], col)}')
        display(HTML('<div style="font-family:monospace;white-space:pre-wrap;line-height:1.5;margin:2px 0 10px 0">'
                     + "\n".join(L) + "</div>"))
    else:   # plain terminal fallback: ANSI colours
        paint = lambda t, col: f"{_ANSI[col]}{t}\033[0m"
        print(paint(head, _GREEN if ok else _RED))
        if error: print("   " + paint("⚠ " + error, _RED))
        for c in checks:
            col = _GREEN if c["ok"] else _RED
            mark = paint("✔" if c["ok"] else "✘", col)
            if c["compact"]:
                print(f'   {mark} {c["label"]}   EXPECTED: {paint(c["exp"], _GREEN)}   ACTUAL: {paint(c["act"], col)}')
            else:
                print(f'   {mark} {c["label"]}')
                print(f'        EXPECTED: {paint(c["exp"], _GREEN)}')
                print(f'        ACTUAL:   {paint(c["act"], col)}')

def _compile(pattern, flags=0, engine=re):
    if engine is None:
        return None, "Optional module missing. Run:  pip install regex   then restart the kernel."
    if not isinstance(pattern, str) or not pattern.strip():
        return None, "Your pattern is empty - fill in the variable above."
    try:
        return engine.compile(pattern, flags), None
    except Exception as e:
        return None, f"Invalid regex: {e}"

def grade_matches(tid, pattern, yes, no, flags=0, how="fullmatch", engine=re):
    rx, err = _compile(pattern, flags, engine)
    checks = []
    for s, want in [(s, True) for s in yes] + [(s, False) for s in no]:
        got = _NA if rx is None else bool(getattr(rx, how)(s))
        checks.append(_chk(repr(s), want, got, fmt=_mm, compact=True))
    _rep(tid, checks, err)

def grade_findall(tid, pattern, cases, flags=0, group=0, engine=re):
    rx, err = _compile(pattern, flags, engine)
    checks = []
    for text, exp in cases:
        got, e = (_NA, None) if rx is None else _try(lambda: [m.group(group) for m in rx.finditer(text)])
        checks.append(_chk(f"text {text!r}", exp, got, e))
    _rep(tid, checks, err)

def grade_findall_raw(tid, pattern, cases, flags=0):
    """Uses re.findall exactly (so capturing-group behaviour matters)."""
    rx, err = _compile(pattern, flags)
    checks = []
    for text, exp in cases:
        got, e = (_NA, None) if rx is None else _try(lambda: rx.findall(text))
        checks.append(_chk(f"text {text!r}", exp, got, e))
    _rep(tid, checks, err)

def grade_sub(tid, pattern, repl, cases, flags=0):
    rx, err = _compile(pattern, flags)
    checks = []
    for text, exp in cases:
        got, e = (_NA, None) if rx is None else _try(lambda: rx.sub(repl, text))
        checks.append(_chk(f"text {text!r}", exp, got, e))
    _rep(tid, checks, err)

def grade_groups(tid, pattern, cases, flags=0):
    rx, err = _compile(pattern, flags)
    checks = []
    for text, exp in cases:
        def run():
            m = rx.search(text)
            return m.groupdict() if m else None
        got, e = (_NA, None) if rx is None else _try(run)
        checks.append(_chk(f"text {text!r}", exp, got, e))
    _rep(tid, checks, err)

def grade_extract(tid, pattern, cases, flags=0):
    """First capturing group of re.search, or None if no match (like pandas .str.extract)."""
    rx, err = _compile(pattern, flags)
    checks = []
    for text, exp in cases:
        def run():
            m = rx.search(text)
            return (m.group(1) if (m and rx.groups) else (m.group(0) if m else None))
        got, e = (_NA, None) if rx is None else _try(run)
        checks.append(_chk(f"text {text!r}", exp, got, e))
    _rep(tid, checks, err)

def grade_func(tid, fn, cases):
    checks = []
    for args, exp in cases:
        if not isinstance(args, tuple): args = (args,)
        got, e = _try(lambda: fn(*args))
        checks.append(_chk("input " + ", ".join(map(repr, args)), exp, got, e))
    _rep(tid, checks)

def grade_fast(tid, pattern, yes, no, slow_input, timeout=3):
    """Semantics via fullmatch + a timing test in a separate process."""
    rx, err = _compile(pattern)
    checks = []
    for s, want in [(s, True) for s in yes] + [(s, False) for s in no]:
        got = _NA if rx is None else bool(rx.fullmatch(s))
        checks.append(_chk(repr(s), want, got, fmt=_mm, compact=True))
    label = f"speed test: fullmatch on a {len(slow_input)}-char tricky input"
    exp = f"finishes in under {timeout}s"
    if rx is not None and all(c["ok"] for c in checks):
        code = f"import re; re.compile({pattern!r}).fullmatch({slow_input!r})"
        t0 = time.perf_counter()
        try:
            subprocess.run([sys.executable, "-c", code], timeout=timeout, capture_output=True)
            took = time.perf_counter() - t0
            checks.append(dict(label=label, exp=exp, act=f"finished in {took:.2f}s", ok=True, compact=True))
        except subprocess.TimeoutExpired:
            checks.append(dict(label=label, exp=exp, act=f"TIMED OUT after {timeout}s: catastrophic backtracking!", ok=False, compact=False))
    else:
        checks.append(dict(label=label, exp=exp, act="— (skipped until the matches above are correct)", ok=False, compact=False))
    _rep(tid, checks, err)

def grade_literal(tid, fn):
    checks = []
    for term, other in [("a.b", "axb"), ("C++", "CC"), ("(x)", "x"), ("1+1=2", "111=2"), ("[tag]", "t")]:
        try: rx, e = re.compile(fn(term)), None
        except Exception as ex: rx, e = None, f"{type(ex).__name__}: {ex}"
        text = f"zz {term} zz"
        g1 = _NA if rx is None else bool(rx.search(text))
        g2 = _NA if rx is None else bool(rx.search(other))
        checks.append(_chk(f"pattern for {term!r} searching {text!r}", True, g1, e, fmt=_mm, compact=True))
        checks.append(_chk(f"pattern for {term!r} searching {other!r}", False, g2, e, fmt=_mm, compact=True))
    _rep(tid, checks)

def scoreboard():
    total = ok = 0
    print("PART".ljust(34), "SCORE", "  TASKS")
    for part, ids in PARTS.items():
        n = sum(RESULTS.get(i) is True for i in ids)
        row = " ".join("✅" if RESULTS.get(i) else ("❌" if RESULTS.get(i) is False else "⬜") for i in ids)
        print(part.ljust(34), f"{n}/{len(ids)}".ljust(5), " ", row)
        total += len(ids); ok += n
    print("-" * 70)
    print(f"TOTAL: {ok}/{total}   ({100*ok//total}%)   ⬜ = not attempted yet")
    if ok == total: print("🏆 Perfect score: regex mastery unlocked!")

PARTS = {
 "Part 1: Foundations": [
  "1.1",
  "1.2",
  "1.3",
  "1.4"
 ],
 "Part 2: Core Building Blocks": [
  "2.1",
  "2.2",
  "2.3",
  "2.4",
  "2.5",
  "2.6",
  "2.7",
  "2.8",
  "2.9",
  "2.10"
 ],
 "Part 3: Advanced Features": [
  "3.1",
  "3.2",
  "3.3",
  "3.4",
  "3.5",
  "3.6"
 ],
 "Part 4: Unicode and Normalization": [
  "4.1",
  "4.2",
  "4.3"
 ],
 "Part 5: NLP Recipes": [
  "5.1",
  "5.2",
  "5.3",
  "5.4",
  "5.5",
  "5.6",
  "5.7",
  "5.8",
  "5.9",
  "5.10",
  "5.11",
  "5.12"
 ],
 "Part 6: The `regex` Module (optional)": [
  "6.1"
 ],
 "Part 7: Performance and Safety": [
  "7.1"
 ],
 "Part 8: Debugging and Testing": [
  "8.1",
  "8.2"
 ],
 "Part 9: Common Pitfalls": [
  "9.1",
  "9.2",
  "9.3"
 ],
 "Part 10: Exercises E1-E8": [
  "E1",
  "E2",
  "E3",
  "E4",
  "E5",
  "E6",
  "E7",
  "E8"
 ],
 "Part 11: Capstone": [
  "C1",
  "C2"
 ]
}
print('Grader ready.')

Grader ready.


---
## Part 1: Foundations

**Recap.** Use raw strings `r"..."`. API: `search` (anywhere), `match` (start), `fullmatch` (whole string), `findall`, `finditer` (Match objects with `.span()`), `sub`, `split`. `.` is any character, so escape it (`\.`) for a literal dot.

### Task 1.1: Literal dot
**Rule:** Write a pattern that matches **exactly** the string `3.14` (and nothing like `3x14`). Graded with `fullmatch`.

<details><summary>Hint</summary>

A bare `.` matches any character.

</details>

In [60]:
t1_1 = r"3\.14"   # <- your regex

grade_matches("1.1", t1_1, yes=["3.14"], no=["3x14", "314", "3.145", "3,14"])

### Task 1.2: Extract integers
**Rule:** Write a pattern so that every run of digits in a text is one match (graded with `finditer`).

In [61]:
t1_2 = r"\d+"

grade_findall("1.2", t1_2, [("Room 12, floor 3, and 405 seats", ["12","3","405"]), ("no digits here", []), ("a1b22c333", ["1","22","333"])])

### Task 1.3: Match spans
**Rule:** Complete `five_letter_spans(text)`: return a list of `(start, end)` character offsets for every **whole word of exactly 5 letters** (letters A-Z/a-z only). Use `re.finditer` and `m.span()`.

<details><summary>Hint</summary>

Combine `\b`, a letter class and `{5}`.

</details>

In [62]:
def five_letter_spans(text):
    ls = re.finditer(r"\b[A-Za-z]{5}\b", text)
    return [match.span() for match in ls]

grade_func("1.3", five_letter_spans, [("hello big world", [(0,5),(10,15)]), ("abcdef abcde", [(7,12)]), ("Ab12c hello_", [])])

### Task 1.4: Escape user input
**Rule:** Complete `literal_pattern(term)`: return a regex string that matches `term` **literally**, even if it contains `. + ( [` etc.

In [63]:
def literal_pattern(term):
    return re.escape(term)

grade_literal("1.4", literal_pattern)

---
## Part 2: Core Building Blocks

**Recap.** Classes `[abc] [^abc] [a-z]`; shorthands `\d \w \s \b`; anchors `^ $ \A \Z`; quantifiers `* + ? {n,m}` (greedy; add `?` for lazy); alternation `|` (longest first); groups `( )`, `(?: )`, `(?P<name> )`; backreferences `\1`, `(?P=name)`.

### Task 2.1: Character classes
**Rule:** Match **runs of consonants** (letters that are not a, e, i, o, u; ignore spaces) in lowercase text.

<details><summary>Hint</summary>

A negated class: `[^...]`.

</details>

In [64]:
t2_1 = r"[^aeiou\s]+"

grade_findall("2.1", t2_1, [("natural language", ["n","t","r","l","l","ng","g"])])

### Task 2.2: Whole words
**Rule:** Match the **whole word** `cat` (lowercase only): not inside `concatenate` or `scatter`, not `Cat`.

In [65]:
t2_2 = r"\bcat\b"

grade_findall("2.2", t2_2, [("cat concatenate cat. Cat scatter", ["cat","cat"]), ("the cat's toy", ["cat"])])

### Task 2.3: Anchors + MULTILINE
**Rule:** Match each **full line that starts with** `TODO` (no leading spaces). The grader applies `re.MULTILINE`.

<details><summary>Hint</summary>

`.` does not cross newlines by default.

</details>

In [66]:
t2_3 = r"^TODO.+$"

grade_findall("2.3", t2_3, [("TODO: a\nDONE: b\nTODO: c\n  TODO: indented", ["TODO: a","TODO: c"])], flags=re.M)

### Task 2.4: Quantifiers
**Rule:** Validate (graded with `fullmatch`) US-style phone numbers of the form `555-123-4567`: 3 digits, dash, 3 digits, dash, 4 digits.

In [67]:
t2_4 = r"(\d{3})-(\d{3})-(\d{4})"

grade_matches("2.4", t2_4, yes=["555-123-4567","000-000-0000"], no=["5551234567","555-12-34567","555-123-45678","a55-123-4567","555-123-4567 "])

### Task 2.5: Greedy vs lazy
**Rule:** Capture the text **inside each pair of double quotes** (use one capturing group; the grader reads group 1).

<details><summary>Hint</summary>

Lazy `.*?` or a negated class `[^"]*`.

</details>

In [68]:
t2_5 = r'"(.*?)"'

grade_findall("2.5", t2_5, [('say "hi" and "there" ok', ["hi","there"]), ('"" empty', [""]), ("none", [])], group=1)

### Task 2.6: Alternation
**Rule:** Match the whole words `cat`, `dog`, `bird` **with an optional plural `s`**. Not `catalog` or `dogma`.

In [69]:
t2_6 = r"\b(cat|dog|bird)s?\b"

grade_findall("2.6", t2_6, [("cats and a dog, birds, catalog, dogma", ["cats","dog","birds"])])

### Task 2.7: Groups
**Rule:** Capture the **surname** after a title `Mr`, `Ms` or `Dr` (optional dot after the title, then a space, then a capitalised word). Use a capturing group for the surname only; group 1 is graded.

In [70]:
t2_7 = r"(?:Mr|Ms|Dr)\.? ([A-Z]\w+)"

grade_findall("2.7", t2_7, [("Dr. Smith met Ms Jones and Mr. Lee", ["Smith","Jones","Lee"]), ("Mister Bob", [])], group=1)

### Task 2.8: Named groups
**Rule:** Parse ISO dates into named groups **year**, **month**, **day**.

In [71]:
t2_8 = r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})"

grade_groups("2.8", t2_8, [("Released 2024-03-15", {"year":"2024","month":"03","day":"15"}), ("on 1999-12-31!", {"year":"1999","month":"12","day":"31"})])

### Task 2.9: Backreference: swap
**Rule:** Turn `Last, First` into `First Last` using `re.sub`. Write the pattern **and** the replacement.

In [72]:
t2_9_pat  = r"\b([A-Za-z]+),\s+([A-Za-z]+)\b"
t2_9_repl = r"\2 \1"

grade_sub("2.9", t2_9_pat, t2_9_repl, [("Doe, John", "John Doe"), ("Smith, Anna and Lee, Bo", "Anna Smith and Bo Lee")])

### Task 2.10: Backreference: repeats
**Rule:** Collapse **consecutive repeated words** (`the the cat` -> `the cat`; any number of repeats). `the theory` must stay unchanged.

In [73]:
t2_10_pat  = r"\b(\w+)(\s+\1)+\b"
t2_10_repl = r"\1"

grade_sub("2.10", t2_10_pat, t2_10_repl, [("the the cat sat sat", "the cat sat"), ("no repeats here", "no repeats here"), ("It is is is fine", "It is fine"), ("the theory", "the theory")])

---
## Part 3: Advanced Features

**Recap.** Lookarounds are zero-width: `(?=..)` `(?!..)` `(?<=..)` `(?<!..)` (lookbehind must be fixed-width in `re`). Flags: `(?i)`, `re.M`, `re.S`, `re.X`. `re.sub` accepts a function.

### Task 3.1: Lookbehind
**Rule:** Match the **number after a dollar sign** without including `$`: integer with optional 2-decimal part (`$20.50` -> `20.50`, `$5` -> `5`). Not euros.

In [74]:
t3_1 = r"(?<=\$)\d+(?:\.\d{2})?"

grade_findall("3.1", t3_1, [("Costs $20.50 or $5, not 30 euros or \u20ac7", ["20.50","5"])])

### Task 3.2: Negative lookahead
**Rule:** Match `foo` only when it is **not followed by** `bar`.

In [75]:
t3_2 = r"\bfoo(?!bar)"

grade_findall("3.2", t3_2, [("foobar foo foobaz", ["foo","foo"]), ("foo", ["foo"]), ("foobar", [])])

### Task 3.3: Lookahead trick
**Rule:** Insert thousands separators: write a pattern that matches the **empty positions** where a comma belongs, and use `,` as replacement. (`1234567` -> `1,234,567`.)

<details><summary>Hint</summary>

Zero-width match: not a boundary, followed by groups of three digits up to a non-digit.

</details>

In [76]:
t3_3 = r"\B(?=(?:\d{3})+(?!\d))"   # replacement is ","

grade_sub("3.3", t3_3, ",", [("1234567 and 89 and 1000", "1,234,567 and 89 and 1,000"), ("999", "999"), ("12345.678", "12,345.678")])

### Task 3.4: Inline flag
**Rule:** Match the word `python` in **any capitalisation** using an **inline flag** (the grader passes no flags).

In [77]:
t3_4 = r"(?i)\bpython\b"

grade_findall("3.4", t3_4, [("Python PYTHON python pythons", ["Python","PYTHON","python"])])

### Task 3.5: Verbose mode
**Rule:** Write a **multi-line, commented** pattern (grader uses `re.VERBOSE`) that validates 24-hour times: `0:00`, `09:05`, `23:59` OK; `24:00`, `12:60`, `1:234` not.

In [78]:
t3_5 = r"""
(?:[01]?[0-9]|2[0-3])
:
[0-5][0-9]
"""

grade_matches("3.5", t3_5, yes=["0:00","09:05","23:59","9:05"], no=["24:00","12:60","12:5","1:234","25:00"], flags=re.VERBOSE)

### Task 3.6: sub with a function
**Rule:** Complete `censor(text)`: replace every run of **12 or more digits** with the same number of `*`; shorter numbers stay.

In [79]:
def censor(text):
    text = re.sub(r"\d{12,}", lambda match: "*" * len(match.group()), text)
    return text

grade_func("3.6", censor, [("card 4111111111111111 on file, id 12345", "card " + "*"*16 + " on file, id 12345"), ("123456789012", "*"*12), ("12345678901", "12345678901")])

---
## Part 4: Unicode and Normalization

**Recap.** Normalize text (`unicodedata.normalize("NFKC", ...)`) before matching. In Python 3, `\w` is Unicode-aware, `[^\W\d_]` means "letters only". The third-party `regex` module adds `\p{L}`, `\p{Lu}`, etc.

### Task 4.1: Normalizer
**Rule:** Complete `normalize(text)`: NFKC-normalize, turn curly quotes into straight ones (`‘ ’` -> `'`, `“ ”` -> `"`), turn any dash `\u2010-\u2015` into `-`, and turn NBSP/odd spaces (`\u00a0`, `\u2000-\u200b`) into a normal space.

In [80]:
def normalize(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u2018\u2019]", "'", text)
    text = re.sub(r"[\u201c\u201d]", '"', text)
    text = re.sub(r"[\u2010-\u2015]", "-", text)
    text = re.sub(r"[\u00a0\u2000-\u200b]", " ", text)
    return text

grade_func("4.1", normalize, [("It\u2019s a \u201csmart\u201d \u2014 caf\u00e9\u00a0idea", "It's a \"smart\" - caf\u00e9 idea"), ("\ufb01nal\u2013draft", "final-draft")])

### Task 4.2: Letters only (Unicode)
**Rule:** Match runs of **letters only** (no digits, no underscore), in any language.

In [81]:
t4_2 = r"[^\W\d_]+"

grade_findall("4.2", t4_2, [("abc_123 d\u00e9f a\u011f", ["abc","d\u00e9f","a\u011f"]), ("Az\u0259rbaycan dili", ["Az\u0259rbaycan","dili"])])

### Task 4.3: BONUS (regex module)
**Rule:** Using the `regex` module: match **capitalised words in any script**: one uppercase letter followed by lowercase letters. Needs `pip install regex`.

### Task 4.2: Letters only (Unicode)
**Rule:** Match runs of **letters only** (no digits, no underscore), in any language.

In [82]:
import regex
t4_3 = r"\p{Lu}\p{Ll}+"

grade_findall("4.3", t4_3, [("\u00c9lan met Zo\u00eb in Bak\u0131", ["\u00c9lan","Zo\u00eb","Bak\u0131"]), ("ALL CAPS here", [])], engine=regex)

---
## Part 5: NLP Recipes

**Recap.** Cleaning pipelines (order matters), tokenizers with named alternatives (`m.lastgroup`), sentence splitting with lookbehinds, entity extractors, gazetteers (`re.escape`, longest first), PII scrubbing.

### Task 5.1: Cleaning pipeline
**Rule:** Complete `clean(text)`: unescape HTML entities, strip HTML tags, replace URLs (`http(s)://...` or `www....`) with `<URL>`, emails with `<EMAIL>`, `@mentions` with `<USER>`, turn `#tag` into `tag`, then collapse whitespace and strip.

<details><summary>Hint</summary>

Remove emails/URLs before mentions, otherwise `@` inside an email gets eaten.

</details>

In [83]:
HTML_TAG_RE = re.compile(r"</?[^>]+>")
URL_RE = re.compile(r"(?:http)s?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
MENTION_RE = re.compile("@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")

def clean(text):
    text = html.unescape(text)                 # &amp; -> &
    text = HTML_TAG_RE.sub("", text)
    text = URL_RE.sub("<URL>", text)
    text = EMAIL_RE.sub("<EMAIL>", text)
    text = MENTION_RE.sub("<USER>", text)
    text = HASHTAG_RE.sub(r"\1", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

grade_func("5.1", clean, [("Loved it!!! <br>Visit https://ex.com/a?b=1 @bob #NLP &amp; more", "Loved it!!! Visit <URL> <USER> NLP & more"), ("Mail bob@ex.com or www.site.org now", "Mail <EMAIL> or <URL> now")])

### Task 5.2: Regex tokenizer
**Rule:** Write `TOKEN_RE` with **named groups in this priority order**: `URL` (`https?://\S+`), `EMAIL`, `HASHTAG`, `MENTION`, `NUMBER` (digits with optional `.`/`,` groups), `WORD` (word chars, may contain inner `'` or `-`), `PUNCT` (any other non-space char). `tokenize` returns `[(type, text), ...]`.

<details><summary>Hint</summary>

In VERBOSE mode escape a literal `#` as `\#`.

</details>

In [84]:
TOKEN_RE = re.compile(r"""
    (?P<URL>      https?://\S+ )
  | (?P<EMAIL>    [\w.+-]+@[\w-]+(?:\.[\w-]+)+ )
  | (?P<HASHTAG>  \#\w+ )
  | (?P<MENTION>  @\w+ )
  | (?P<NUMBER>   \d+(?:[.,]\d+)* )
  | (?P<WORD>     \w+(?:['’-]\w+)* )
  | (?P<PUNCT>    [^\w\s] )
""", re.VERBOSE)

def tokenize(text):
    return [(m.lastgroup, m.group()) for m in TOKEN_RE.finditer(text)]

grade_func("5.2", tokenize, [("Email bob@ex.com about the state-of-the-art model, it's $3.50! #nlp", [("WORD","Email"),("EMAIL","bob@ex.com"),("WORD","about"),("WORD","the"),("WORD","state-of-the-art"),("WORD","model"),("PUNCT",","),("WORD","it's"),("PUNCT","$"),("NUMBER","3.50"),("PUNCT","!"),("HASHTAG","#nlp")]), ("Visit https://x.org/a now @alice", [("WORD","Visit"),("URL","https://x.org/a"),("WORD","now"),("MENTION","@alice")])])

### Task 5.3: Sentence splitter
**Rule:** Complete `split_sentences(text)`: split after `.`, `!`, `?` followed by whitespace and a capital letter, but **not after** `Mr.`, `Mrs.`, `Dr.`, `Prof.`. Strip pieces.

In [85]:
SENT_SPLIT = re.compile(r"""
    (?<!\bMr\.)(?<!\bMrs\.)(?<!\bDr\.)(?<!\bProf\.)(?<!\be\.g\.)(?<!\bi\.e\.)
    (?<=[.!?])\s+(?=["“(]?[A-Z])
""", re.VERBOSE)

def split_sentences(text):
    ls = SENT_SPLIT.split(text)
    return ls

grade_func("5.3", split_sentences, [("Dr. Smith arrived. He was late! Was it traffic? Probably.", ["Dr. Smith arrived.","He was late!","Was it traffic?","Probably."]), ("Mr. Lee said hi. Bye.", ["Mr. Lee said hi.","Bye."]), ("Use tools e.g. regex. It works.", ["Use tools e.g. regex.","It works."])])

### Task 5.4: Contractions
**Rule:** Split contractions Penn-Treebank style with `re.sub`: `can't` -> `ca n't`, `they're` -> `they 're`, `it's` -> `it 's` (suffixes `n't 's 're 've 'll 'd 'm`).

<details><summary>Hint</summary>

Lazy `\w+?` so that the suffix `n't` is split off.

</details>

In [86]:
t5_4_pat  = r"\b(\w+?)(n't|'s|'re|'ve|'ll|'d|'m)\b"
t5_4_repl = r"\1 \2"

grade_sub("5.4", t5_4_pat, t5_4_repl, [("I can't believe they're here and it's late", "I ca n't believe they 're here and it 's late"), ("we don't and you'll see", "we do n't and you 'll see")], flags=re.I)

### Task 5.5: Elongated words
**Rule:** Cap letter repetition at two: `sooooo goooood!!!` -> `soo good!!!` (any word character repeated 3+ times becomes 2).

In [87]:
t5_5_pat  = r"(\w)\1{2,}"
t5_5_repl = r"\1\1"

grade_sub("5.5", t5_5_pat, t5_5_repl, [("sooooo goooood!!!", "soo good!!!"), ("coool", "cool"), ("book", "book")])

### Task 5.6: Money extractor
**Rule:** Match money amounts: a currency symbol (`$ € £`), optional space, digits with optional thousands commas, optional 2-digit decimals. (`99 dollars` is NOT matched.)

In [88]:
t5_6 = r"[$€£]\d+(?:,\d{3})*(?:\.\d{2})?"

grade_findall("5.6", t5_6, [("Paid $1,250.00 and \u20ac30 plus \u00a37.50, not 99 dollars", ["$1,250.00","\u20ac30","\u00a37.50"])])

### Task 5.7: Percent extractor
**Rule:** Match percentages like `15%`, `3.5 %`, `100%` (number, optional single space, `%`).

In [89]:
t5_7 = r"\d+(?:\.\d+)?\s?%"

grade_findall("5.7", t5_7, [("15% off, 3.5 % rise, 100%", ["15%","3.5 %","100%"]), ("no percent 50", [])])

### Task 5.8: Email validator
**Rule:** Pragmatic email check with `fullmatch`: local part (letters, digits, `_ . + -`), `@`, domain labels separated by dots, **at least one dot**.

In [90]:
t5_8 = r"[A-Za-z0-9_.+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+"

grade_matches("5.8", t5_8, yes=["a@b.org","john.doe+tag@mail.example.co.uk"], no=["a@b","@b.org","a b@c.org","a@b..org","a@.org"])

### Task 5.9: Gazetteer builder
**Rule:** Complete `build_gazetteer(terms)` returning a compiled regex that finds any term as a **whole word**, **case-insensitively**, preferring the **longest** term, safe for special characters.

In [91]:
def build_gazetteer(terms):
    escaped_terms = [re.escape(term) for term in terms]
    pattern = r"\b(?:" + "|".join(sorted(escaped_terms, key=len, reverse=True)) + r")\b"
    return re.compile(pattern, re.IGNORECASE)

TERMS = ["New York", "New York City", "Los Angeles", "San Francisco", "St. Louis"]
grade_func("5.9", lambda t: [m.group() for m in build_gazetteer(TERMS).finditer(t)], [("Flights from new york city to Los Angeles, not Newark; also New York.", ["new york city","Los Angeles","New York"]), ("St. Louis but not Stx Louis", ["St. Louis"])])

### Task 5.10: Extract with a group (pandas style)
**Rule:** Extract the **order number** after `#` using one capturing group (same idea as `df.str.extract`). No `#digits` -> no match.

In [92]:
t5_10 = r"#(\d+)"

grade_extract("5.10", t5_10, [("Order #123 on 2024-03-15", "123"), ("no id", None), ("Order #77 on 2023-12-01", "77")])

### Task 5.11: Product codes
**Rule:** Validate product codes with `fullmatch`: **two uppercase letters, dash, four digits** (`AB-1234`).

In [93]:
t5_11 = r"([A-Z]{2})-(\d{4})"

grade_matches("5.11", t5_11, yes=["AB-1234","ZZ-0000"], no=["ab-1234","ABC-1234","AB-123","AB-12345","AB1234"])

### Task 5.12: PII scrubber
**Rule:** Complete `scrub(text)`: replace US SSNs (`123-45-6789`) with `[SSN]`, emails with `[EMAIL]`, then phone numbers (optional `+`, digits and `() . - space`, 10+ chars, starting and ending with a digit) with `[PHONE]`. Order matters!

In [94]:
SSN_PATTERN_RE = re.compile(r"(\d{3})-(\d{2})-(\d{4})")
EMAIL_PATTERN_RE = re.compile(r"\b([A-Za-z0-9_\.\+\-]+)@([A-Za-z0-9-]+)(?:\.[A-Za-z0-9-]+)+")
PHONE_PATTERN_RE = re.compile(r"(?<!\w)\+?\d(?:[\d(). -]*\d){5,}(?!\w)")

def scrub(text):
    text = SSN_PATTERN_RE.sub("[SSN]", text)
    text = EMAIL_PATTERN_RE.sub("[EMAIL]", text)
    text = PHONE_PATTERN_RE.sub("[PHONE]", text)
    return text

grade_func("5.12", scrub, [("SSN 123-45-6789, mail a.b@c.com, call +1 (555) 123-4567.", "SSN [SSN], mail [EMAIL], call [PHONE]."), ("nothing here 42", "nothing here 42")])

---
## Part 6: The `regex` Module (optional)

**Recap.** `pip install regex`: variable-width lookbehind, `\p{...}` properties, fuzzy matching `{e<=1}`, `overlapped=True`, recursion `(?R)`.

### Task 6.1: BONUS: variable-width lookbehind
**Rule:** With the `regex` module, match the **surname after** `Mr`, `Mrs` or `Dr` with an optional dot, using a **lookbehind** (so the title is not in the match).

In [95]:
t6_1 = r"(?<=(?:Mr|Mrs|Dr)\.?\s)\w+"

grade_findall("6.1", t6_1, [("Mr. Smith and Dr Jones and Smith", ["Smith","Jones"])], engine=regex)

---
## Part 7: Performance and Safety

**Recap.** Nested quantifiers like `(a+)+` explode on near-misses (ReDoS). Rewrite to an equivalent, unambiguous pattern.

### Task 7.1: Fix catastrophic backtracking
**Rule:** The pattern `^(a+)+$` is **catastrophically slow** on `aaaa...ab`. Write an **equivalent fast** pattern (graded with `fullmatch`; there is a 3-second timing test).

<details><summary>Hint</summary>

`(a+)+` accepts the same strings as plain `a+`.

</details>

In [96]:
t7_1 = r"a+"

grade_fast("7.1", t7_1, yes=["a","aaaa"], no=["","aab","b"], slow_input="a"*40 + "b")

---
## Part 8: Debugging and Testing

**Recap.** `$` matches before a trailing newline; `\Z` does not. `findall` returns tuples when the pattern has several groups.

### Task 8.1: The trailing-newline bug
**Rule:** Fix the pattern `\d+$` so that it rejects `"123\n"`. The grader uses `re.match` (not `fullmatch`).

In [97]:
t8_1 = r"\d+\Z"   # <- buggy, fix it

grade_matches("8.1", t8_1, yes=["123","7"], no=["123\n","12a","","a1"], how="match")

### Task 8.2: findall returns tuples?
**Rule:** Rewrite `(\d+)-(\d+)` so that `re.findall` returns the **whole matches** (`['10-20', '3-4']`) instead of tuples.

In [98]:
t8_2 = r"(\d+-\d+)"   # <- fix it

grade_findall_raw("8.2", t8_2, [("10-20 and 3-4", ["10-20","3-4"]), ("none", [])])

---
## Part 9: Common Pitfalls

**Recap.** `match` anchors at the start only; always `re.escape` literals; greedy dots overreach; `[^\W\d_]` = letters only.

### Task 9.1: match vs search
**Rule:** With `re.match` (anchored at the start!) a pattern must **find `ERROR` anywhere** in the line. Write it.

In [99]:
t9_1 = r".*ERROR"

grade_matches("9.1", t9_1, yes=["ERROR x","a ERROR","aERRORb"], no=["ok","error","WARN"], how="match")

### Task 9.2: Counting literals
**Rule:** Complete `count_literal(term, text)`: count **literal** occurrences of `term` in `text` using regex (watch special characters!).

In [100]:
def count_literal(term, text):
    return len(re.findall(re.escape(term), text))

grade_func("9.2", count_literal, [(("a.b","a.b axb a.b"), 2), (("C++","C++ and C++ C"), 2), (("(x)","f(x) g(x) x"), 2)])

### Task 9.3: Strip tags
**Rule:** Remove HTML tags with `re.sub` (replace by empty string). Avoid the greedy-dot trap.

In [101]:
t9_3 = r"</?[^>]+>"   # replacement is ""

grade_sub("9.3", t9_3, "", [("<b>bold</b> and <i>x</i>", "bold and x"), ('<p class="x">Hi</p>', "Hi")])

---
## Part 10: Exercises E1-E8

Classic NLP mini-problems from the tutorial.

### Task E1: Hashtags
**Rule:** Complete `hashtags(text)`: return all hashtags **without `#`, lowercased**.

In [102]:
def hashtags(text):
    return [tag.lower() for tag in re.findall(r"#(\w+)", text)]

grade_func("E1", hashtags, [("Loving #NLP and #DeepLearning!", ["nlp","deeplearning"]), ("no tags", []), ("#a#b", ["a","b"])])

### Task E2: Ordinal dates
**Rule:** Validate dates like `March 3rd, 2024` (full English month name, day 1-2 digits + `st|nd|rd|th`, comma, 4-digit year). `fullmatch`.

In [103]:
t_e2 = r"(January|February|March|April|May|June|July|August|September|October|November|December) \d{1,2}(st|nd|rd|th), \d{4}"


grade_matches("E2", t_e2, yes=["March 3rd, 2024","May 21st, 1999","December 12th, 2030"], no=["March 3, 2024","Marc 3rd, 2024","March 3rd 2024","March 3rd, 24"])

### Task E3: snake_case <-> camelCase
**Rule:** Complete `to_camel(s)` (`my_var_name` -> `myVarName`) and `to_snake(s)` (`myVarName` -> `my_var_name`) with `re.sub`.

In [104]:
def to_camel(s):
    return re.sub(r'_([a-z])', lambda m: m.group(1).upper(), s)

def to_snake(s):
    return re.sub(r'([A-Z])', lambda m: '_' + m.group(1).lower(), s).lstrip('_')

grade_func("E3", lambda a, b: (to_camel(a), to_snake(b)), [(("my_var_name","myVarName"), ("myVarName","my_var_name")), (("x","x"), ("x","x")), (("a_b","Already"), ("aB","already"))])

### Task E4: Duplicate sentences
**Rule:** Remove **consecutively repeated sentences** with one `re.sub` (`Hi there. Hi there. Bye.` -> `Hi there. Bye.`).

In [105]:
t_e4_pat  = r'(.+?[.!?])(?:\s+\1)+'
t_e4_repl = r'\1'

grade_sub("E4", t_e4_pat, t_e4_repl, [("Hi there. Hi there. Bye.", "Hi there. Bye."), ("Go! Go! Go! Stop.", "Go! Stop."), ("No repeats. Here.", "No repeats. Here.")])

### Task E5: Quoted speech
**Rule:** Capture **(quote, speaker)** pairs for `"..." said Name` using two capture groups (graded with `re.findall`, so tuples are expected).

In [106]:
t_e5 = r'"([^"]+)"\s+said\s+([A-Z][a-z]+)'

grade_findall_raw("E5", t_e5, [('"Run" said Tom. "Stop" said Ann.', [("Run","Tom"),("Stop","Ann")]), ("no quotes said Bob", [])])

### Task E6: Trim punctuation
**Rule:** Strip leading and trailing punctuation but keep inner apostrophes/hyphens: `'don't-stop!!` -> `don't-stop`. One pattern, replacement is `""`.

In [107]:
t_e6 = r'^[^\w\s]+|[^\w\s]+$'

grade_sub("E6", t_e6, "", [("'don't-stop!!", "don't-stop"), ("...hello, world?!", "hello, world")])

### Task E7: Mask card digits
**Rule:** Mask every digit **except the last four** with `*` (`4111-1111-1111-1234` -> `****-****-****-1234`). Pattern matches **one digit**, replacement is `*`.

<details><summary>Hint</summary>

Digit followed (after any non-digits) by at least four more digits.

</details>

In [108]:
t_e7 = r"\d(?=.*\d{4})"

grade_sub("E7", t_e7, "*", [("4111-1111-1111-1234", "****-****-****-1234"), ("12345", "*2345"), ("1234", "1234")])

### Task E8: Challenge: abbreviation-aware tokenizer
**Rule:** Complete `tokenize2(text)` returning **strings**: keep abbreviations like `U.S.A.` and `Ph.D.` as single tokens, numbers like `3.5` as one token, words with inner `'`/`-`, and every other symbol alone. Abbreviation alternative must come **first**.

In [109]:
def tokenize2(text):
    text = re.sub(r'\b(?:[A-Za-z]+\.){2,}',
                  lambda m: m.group().replace('.', '<DOT>'), text)
    text = re.sub(r'([,!?])', r' \1 ', text)
    text = re.sub(r'(?<!\d)\.(?!\d)', ' . ', text)
    text = text.replace('<DOT>', '.')
    return text.split()

grade_func("E8", tokenize2, [("She has a Ph.D. from the U.S.A. and scored 3.5 points.", ["She","has","a","Ph.D.","from","the","U.S.A.","and","scored","3.5","points","."]), ("It's state-of-the-art, right?", ["It's","state-of-the-art",",","right","?"])])

---
## Part 11: Capstone

Put everything together.

### Task C1: Log line parser
**Rule:** Parse lines like `2024-03-15 14:32:10 [ERROR] user=alice ip=10.0.0.12 msg="Disk full"` into named groups: **date, time, level, user, ip, msg** (msg without quotes). Use `re.VERBOSE` freely.

<details><summary>Hint</summary>

In VERBOSE mode a literal space must be written `\ ` or `[ ]`.

</details>

In [110]:
t_c1 = r"""
(?P<date>\d{4}-\d{2}-\d{2})
[ ]
(?P<time>\d{2}:\d{2}:\d{2})
[ ]\[
(?P<level>[A-Z]+)
\][ ]
user=(?P<user>\w+)
[ ]ip=(?P<ip>\d{1,3}(?:\.\d{1,3}){3})
[ ]msg="(?P<msg>[^"]*)"
"""

grade_groups("C1", t_c1, [('2024-03-15 14:32:10 [ERROR] user=alice ip=10.0.0.12 msg="Disk full"', {"date":"2024-03-15","time":"14:32:10","level":"ERROR","user":"alice","ip":"10.0.0.12","msg":"Disk full"}), ('x 2023-01-02 00:00:01 [INFO] user=bob_1 ip=192.168.1.1 msg="" y', {"date":"2023-01-02","time":"00:00:01","level":"INFO","user":"bob_1","ip":"192.168.1.1","msg":""})], flags=re.VERBOSE)

### Task C2: Mini expression lexer
**Rule:** Complete `lex(expr)` returning `[(type, text), ...]` with types `NUM` (ints/decimals), `ID` (identifier), `OP` (`+ - * / ^`), `LPAREN`, `RPAREN`. Skip whitespace.

In [111]:
def lex(expr):
    pattern = re.compile(
        r'(?P<NUM>\d+(?:\.\d+)?)'
        r'|(?P<ID>[A-Za-z_]\w*)'
        r'|(?P<OP>[+\-*/^])'
        r'|(?P<LPAREN>\()'
        r'|(?P<RPAREN>\))'
    )

    result = []
    for m in pattern.finditer(expr):
        result.append((m.lastgroup, m.group()))

    return result

grade_func("C2", lex, [("3.5 + x1*(2 - 4)", [("NUM","3.5"),("OP","+"),("ID","x1"),("OP","*"),("LPAREN","("),("NUM","2"),("OP","-"),("NUM","4"),("RPAREN",")")]), ("a^2", [("ID","a"),("OP","^"),("NUM","2")])])

---
## Final scoreboard

In [112]:
scoreboard()

PART                               SCORE   TASKS
Part 1: Foundations                4/4     ✅ ✅ ✅ ✅
Part 2: Core Building Blocks       10/10   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
Part 3: Advanced Features          6/6     ✅ ✅ ✅ ✅ ✅ ✅
Part 4: Unicode and Normalization  3/3     ✅ ✅ ✅
Part 5: NLP Recipes                12/12   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
Part 6: The `regex` Module (optional) 1/1     ✅
Part 7: Performance and Safety     1/1     ✅
Part 8: Debugging and Testing      2/2     ✅ ✅
Part 9: Common Pitfalls            3/3     ✅ ✅ ✅
Part 10: Exercises E1-E8           8/8     ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
Part 11: Capstone                  2/2     ✅ ✅
----------------------------------------------------------------------
TOTAL: 52/52   (100%)   ⬜ = not attempted yet
🏆 Perfect score: regex mastery unlocked!
